In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import time
from matplotlib.colors import LinearSegmentedColormap
import warnings

# Ignore warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. Global Font and Style Configuration (Strictly Arial + LaTeX)
# ==========================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False 
plt.rcParams['mathtext.fontset'] = 'stix'

# ==========================================
# Size and Style Core Control Hub (Oversized System)
# ==========================================
CONFIG = {
    'fig_w': 14,             
    'fig_h': 12,             
    'dpi': 600,              

    'label_size': 32,        
    'tick_size': 22,         
    
    'cbar_label_size': 26,   
    'cbar_tick_size': 24,    

    'border_width': 2,       
    'tick_width': 2,         
    'tick_length': 8,        

    'xlabel_pad': 10,        
    'ylabel_pad': 15,        
    'cbar_pad': 15,          
}

# ==========================================
# 2. Simulation Parameter Settings
# ==========================================
S0_total, R0_total = 1e7, 1e6
d, K_cap = 0.07, 1e9
dt, total_time = 0.1, 12 
time_steps = int(total_time / dt)

Vsmax_range, Vrmax_range = (0.98, 1.0), (0.89, 0.90)
Vsmin, Vrmin = -0.1, -0.1
MICr_range, Kr_range = (64.0, 256.0), (1.0, 2.0)
Ks_range = (4.0, 6.0)

DELTA_MIN, DELTA_MAX = 0.03, 0.12  
GAMMA_IB_CAP = 1.40
GAMMA_INTRA_RANGE = (0.95, 1.05)

I_GLOBAL_MIN, I_GLOBAL_MAX = 5, 50
EXTREME_ALPHA = 3.0 

c_resolution = 300
c_range = np.linspace(0, 100, c_resolution)

# ==========================================
# 3. Axis Construction
# ==========================================
def generate_linked_ij_axis():
    i_list = []
    j_list = []
    i_list.extend(range(5, 13)); j_list.extend(np.linspace(1, 2, 8).round().astype(int))
    i_list.extend(range(13, 29)); j_list.extend(np.linspace(2, 3, 16).round().astype(int))
    i_list.extend(range(29, 51)); j_list.extend(np.linspace(4, 5, 22).round().astype(int))
    return np.array(i_list), np.array(j_list)

H_I_SEQ, H_J_SEQ = generate_linked_ij_axis()
H_STEPS = len(H_I_SEQ)
GAMMA_STEPS = 40
gamma_ja_axis = np.linspace(0.8, 1.2, GAMMA_STEPS) 

# ==========================================
# 4. Core Simulation Function
# ==========================================
def calculate_shannon_h(abundance):
    total = np.sum(abundance)
    if total == 0: return 0
    p = abundance / total
    p = p[p > 0]
    return -np.sum(p * np.log(p))

def sample_MICs_extreme_high(i_s, low=2.0, high=8.0, alpha=EXTREME_ALPHA):
    x = (i_s - I_GLOBAL_MIN) / (I_GLOBAL_MAX - I_GLOBAL_MIN)
    x = float(np.clip(x, 0.0, 1.0))
    pool_size = int(np.ceil(i_s * (1.0 + alpha * x)))
    pool = np.random.uniform(low, high, pool_size)
    return np.sort(pool)[-i_s:] 

def run_heatmap_simulation():
    REPEATS = 300 
    print(f"--- Generating clean heatmap (X=H, Y=Gamma) ---")
    print(f"--- Simulations: {REPEATS} per grid ---")
    
    msc_grid = np.zeros((GAMMA_STEPS, H_STEPS))
    h_labels_list = [] 
    
    for h_idx in range(H_STEPS):
        current_i = H_I_SEQ[h_idx]
        current_j = H_J_SEQ[h_idx]
        
        s_abund = np.full(current_i, S0_total/current_i)
        r_abund = np.full(current_j, R0_total/current_j)
        h_val = calculate_shannon_h(np.concatenate([s_abund, r_abund]))
        h_labels_list.append(h_val)
        
        for g_idx, g_ja in enumerate(gamma_ja_axis):
            msc_accum = []
            for _ in range(REPEATS):
                delta = np.random.uniform(DELTA_MIN, DELTA_MAX)
                g_ib = min(g_ja + delta, GAMMA_IB_CAP)
                g_ia = np.random.uniform(*GAMMA_INTRA_RANGE)
                g_jb = np.random.uniform(*GAMMA_INTRA_RANGE)
                
                mics = sample_MICs_extreme_high(current_i)
                micr = np.random.uniform(*MICr_range, current_j)
                ks = np.random.uniform(*Ks_range, current_i)
                kr = np.random.uniform(*Kr_range, current_j)
                vsmax = np.random.uniform(*Vsmax_range, current_i)
                vrmax = np.random.uniform(*Vrmax_range, current_j)
                
                term_s = (c_range[:, None] / mics) ** ks
                vs_mat = vsmax - ((vsmax - Vsmin) * term_s) / (term_s - (Vsmin / vsmax))
                term_r = (c_range[:, None] / micr) ** kr
                vr_mat = vrmax - ((vrmax - Vrmin) * term_r) / (term_r - (Vrmin / vrmax))
                
                S = np.full((c_resolution, current_i), S0_total/current_i)
                R = np.full((c_resolution, current_j), R0_total/current_j)
                
                for _t in range(time_steps):
                    pop_s = np.sum(S, axis=1, keepdims=True)
                    pop_r = np.sum(R, axis=1, keepdims=True)
                    comp_s = 1 - (g_ia * pop_s + g_ja * pop_r) / K_cap
                    comp_r = 1 - (g_ib * pop_s + g_jb * pop_r) / K_cap
                    S += (vs_mat * S * comp_s - d * S) * dt
                    R += (vr_mat * R * comp_r - d * R) * dt
                    S[S<0]=0; R[R<0]=0
                
                tot_s = np.sum(S, axis=1)
                tot_r = np.sum(R, axis=1)
                sc = (tot_s / S0_total) / (tot_r / R0_total + 1e-20)
                try:
                    val = np.interp(1.0, sc[::-1], c_range[::-1])
                    msc_accum.append(val)
                except: pass
            
            if msc_accum:
                msc_grid[g_idx, h_idx] = np.mean(msc_accum)
            else:
                msc_grid[g_idx, h_idx] = np.nan
        
        if h_idx % 5 == 0: print(f"Progress: Col {h_idx}/{H_STEPS} Done")
            
    return msc_grid, np.array(h_labels_list)

msc_results, h_axis = run_heatmap_simulation()

# ==========================================
# 5. Plotting and Saving (Clean version without dashed lines)
# ==========================================
fig = plt.figure(figsize=(CONFIG['fig_w'], CONFIG['fig_h']), dpi=CONFIG['dpi'])
ax = fig.add_axes([0.12, 0.12, 0.72, 0.83])

v_min, v_max = np.nanmin(msc_results), np.nanmax(msc_results)

# Monet Garden gradient: Purple -> Yellow -> Green
colors = ["#2E1E34", "#6B4C68", "#B58AA5", "#D9E3A6", "#A3B69C"]
custom_cmap = LinearSegmentedColormap.from_list("Monet_Garden", colors)

extent = [h_axis[0], h_axis[-1], gamma_ja_axis[0], gamma_ja_axis[-1]]

im = ax.imshow(msc_results, 
               extent=extent, 
               origin='lower', 
               aspect='auto', 
               cmap=custom_cmap, 
               interpolation='nearest',
               vmin=v_min, vmax=v_max)

# --- Label Settings ---
ax.set_xlabel("$H$", fontsize=CONFIG['label_size'], labelpad=CONFIG['xlabel_pad'])
ax.set_ylabel(r"$\gamma_{ja}$", fontsize=CONFIG['label_size'], labelpad=CONFIG['ylabel_pad'])

# --- Tick Settings ---
y_ticks = np.linspace(0.8, 1.2, 5)
ax.set_yticks(y_ticks)
ax.set_yticklabels([f"{y:.1f}" for y in y_ticks])

x_ticks = np.linspace(h_axis[0], h_axis[-1], 6)
ax.set_xticks(x_ticks)
ax.set_xticklabels([f"{x:.2f}" for x in x_ticks])

# --- Style Adjustments ---
ax.tick_params(axis='both', which='major', labelsize=CONFIG['tick_size'], 
               width=CONFIG['tick_width'], length=CONFIG['tick_length'], direction='out')
for spine in ax.spines.values(): spine.set_linewidth(CONFIG['border_width'])

# --- Colorbar ---
cbar_ax = fig.add_axes([0.88, 0.12, 0.025, 0.83]) 
cbar = fig.colorbar(im, cax=cbar_ax)

cbar_ticks = np.linspace(v_min, v_max, 7)
cbar.set_ticks(cbar_ticks)
cbar.ax.set_yticklabels([f'{x:.1f}' for x in cbar_ticks], fontname='Arial', fontsize=CONFIG['cbar_tick_size'])

cbar.set_label('MSC Value', fontsize=CONFIG['cbar_label_size'], labelpad=15)
cbar.ax.tick_params(width=CONFIG['tick_width'], length=CONFIG['tick_length'])
cbar.outline.set_linewidth(CONFIG['border_width'])

# --- Save ---
save_dir = "./results_ecological"
save_path = os.path.join(save_dir, "MSC_Heatmap_Clean.png")
if not os.path.exists(save_dir): os.makedirs(save_dir)

plt.savefig(save_path, dpi=CONFIG['dpi'])
print(f"Figure successfully saved to: {save_path}")

plt.show()